# Полная проверка train: что проверяем и что нашли

**Файл:** data/raw/public-train-annotated.jsonl (исходный train из новой структуры проекта). **Запуск:** выбери Python 3 → Restart Kernel → Run All.
Ноутбук независим от общего ноутбука и test. Для проверок хватает стандартной библиотеки Python;
для интерфейса нужен Jupyter / поддержка ipynb в IDE. Установка при необходимости: python -m pip install jupyter ipykernel.

Сканируем **все строки**, без выборки. Каждый раздел: **цель → правило → результат → что делать**.
Число примеров ограничивает только отображение, а не проверку. Полные результаты сохраняются в конце.
Карточки объявлений раскрываются нажатием на заголовок и содержат **весь текст без обрезки**.

**Границы:** train размечен автоматически и не является проверенным вручную эталоном.
Структуру и точные дубликаты проверяем программно; корректность смысла всех меток регулярными выражениями доказать нельзя.
«Проверить» не означает «ошибка». Эмодзи, пустой набор меток и все пять меток допустимы.
Исходник и метки не меняются. Test не используется для подбора правил.

Порядок: чтение → схема → ID → тексты → метки → дубликаты → артефакты → словарь →
контекст → обрезка → утечка при split → ручная проверка → выгрузки → итог.


## 1. Настройки и удобный вывод

TRAIN_PATH_OVERRIDE — свой путь к файлу; по умолчанию читаем data/raw/public-train-annotated.jsonl.
Корень проекта ищется от текущей папки вверх: запуск работает из корня и из dataset_review.
Копия в innoglobalhack_avito автоматически не используется. data/processed/train_clean.parquet — результат
предобработки; проверка исходной схемы и разметки проводится на raw JSONL.
EXAMPLES_PER_CHECK — число полных примеров под проверкой.
TABLE_PREVIEW_ROWS=None показывает таблицы целиком; большие таблицы удобнее смотреть через show_table.
EXPORT_RESULTS=True сохраняет все результаты в новую папку запуска, сохраняя предыдущую ручную работу.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv, hashlib, html, itertools, json, math, platform, random, re, statistics, sys, unicodedata
try:
    from IPython.display import display, HTML
except ImportError:
    class HTML:
        def __init__(self, data): self.data = data
    def display(value): print(value.data if isinstance(value, HTML) else value)

TRAIN_PATH_OVERRIDE = None
EXAMPLES_PER_CHECK = 2
TABLE_PREVIEW_ROWS = 15
EXPORT_RESULTS = True
RANDOM_SEED = 42
EXPECTED_ROWS = 10_000  # число из README, не ограничение чтения
CATEGORIES = {102:'Штукатурные работы',103:'Плиточные работы',104:'Сантехника',
              105:'Электрика',106:'Поклейка обоев и малярные работы'}
# Корень проекта не зависит от выбранного вручную исходника.
root_candidates = [Path.cwd(), *Path.cwd().parents]
if TRAIN_PATH_OVERRIDE:
    override_path = Path(TRAIN_PATH_OVERRIDE).expanduser().resolve()
    root_candidates += list(override_path.parents)
PROJECT_ROOT = next((p for p in root_candidates
    if (p/'dataset_review').is_dir() and (p/'data/raw').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Запусти ноутбук из проекта: нужны папки dataset_review и data/raw.')
TRAIN_PATH = (Path(TRAIN_PATH_OVERRIDE).expanduser().resolve() if TRAIN_PATH_OVERRIDE
              else PROJECT_ROOT/'data/raw/public-train-annotated.jsonl')
if not TRAIN_PATH.is_file():
    raise FileNotFoundError(f'Нет файла {TRAIN_PATH}. Положи train в data/raw или укажи TRAIN_PATH_OVERRIDE.')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR = PROJECT_ROOT/'dataset_review/train_audit_runs'/RUN_ID
checks, findings, tables, by_line = {}, [], {}, {}
def esc(value): return html.escape(str(value))

def table(rows, columns=None, limit=TABLE_PREVIEW_ROWS):
    rows = list(rows)
    columns = columns or (list(rows[0]) if rows else [])
    shown = rows if limit is None else rows[:limit]
    head = ''.join('<th style="text-align:left;padding:7px">'+esc(c)+'</th>' for c in columns)
    body = ''.join('<tr>'+''.join('<td style="vertical-align:top;white-space:pre-wrap;'
        'overflow-wrap:anywhere;padding:7px;border-bottom:1px solid #ddd">'+esc(row.get(c,''))+'</td>'
        for c in columns)+'</tr>' for row in shown)
    display(HTML(f'<div style="overflow:auto;max-height:650px"><table><thead><tr>{head}</tr></thead>'
                 f'<tbody>{body}</tbody></table></div>'))
    print(f'Показано {len(shown)} из {len(rows)} строк таблицы.' if rows else 'Совпадений нет.')

def remember(name, rows, limit=TABLE_PREVIEW_ROWS):
    tables[name] = list(rows)
    print('Таблица:',name)
    table(tables[name],limit=limit)

def show_table(name,start=0,limit=20):
    rows = tables[name]
    print(f'{name}: всего {len(rows)}, начало страницы {start}.')
    table(rows[start:] if limit is None else rows[start:start+limit],limit=None)

def bars(title,pairs):
    pairs = list(pairs)
    maximum = max([v for _,v in pairs]+[1])
    parts = [f'<h4>{esc(title)}</h4>']
    for label,value in pairs:
        parts.append(f'<div style="display:flex;align-items:center;gap:10px;margin:5px 0">'
            f'<span style="width:270px">{esc(label)}</span>'
            f'<span style="background:#2563eb;height:16px;width:{max(1,330*value/maximum):.1f}px"></span>'
            f'<b>{value:,}</b></div>')
    display(HTML(''.join(parts)))

def show_ad(line_no,expanded=True):
    line_no = int(line_no)
    row = by_line.get(line_no)
    if row is None:
        print('Для этой физической строки нет JSON-объекта; см. parse_issues.'); return
    flags = sorted({f['check'] for f in findings if f['line_no']==line_no})
    heading = f"Строка {line_no} · item_id={row.get('item_id')} · {row.get('title')}"
    labels = json.dumps(row.get('found_microcategories'),ensure_ascii=False,indent=2)
    display(HTML(f'<details {"open" if expanded else ""} style="border:1px solid #aaa;padding:12px;margin:8px 0">'
        f'<summary><b>{esc(heading)}</b></summary><p>Проверки: {esc(", ".join(flags) or "нет срабатываний")}</p>'
        f'<pre style="white-space:pre-wrap">Метки: {esc(labels)}</pre>'
        f'<div style="white-space:pre-wrap;overflow-wrap:anywhere">{esc(row.get("description"))}</div></details>'))

def show_rows(start=0,limit=5):
    numbers = sorted(by_line)
    selected = numbers[start:] if limit is None else numbers[start:start+limit]
    print(f'Полные тексты: {len(selected)} из {len(numbers)}, начало {start}.')
    for n in selected: show_ad(n,expanded=False)

def register(key,title,level,events,scope,rule,action):
    findings[:] = [f for f in findings if f['check']!=key]
    events = list(events)
    findings.extend({'check':key,'level':level,**e} for e in events)
    numbers = sorted({e['line_no'] for e in events})
    checks[key] = dict(check=key,title=title,level=level,checked=scope,
        affected_rows=len(numbers),events=len(events),rule=rule,action=action)
    print(f'\n{title} [{key}] — {level}')
    print(f'Проверено: {scope:,}; строк со срабатыванием: {len(numbers):,}; событий: {len(events):,}.')
    print('Правило:',rule,'\nДействие:',action)
    table(events)
    for n in numbers[:EXAMPLES_PER_CHECK]: show_ad(n,expanded=False)

def show_check(key,start=0,limit=5):
    selected = [f for f in findings if f['check']==key]
    print(checks[key]['title'],'|',checks[key]['rule'])
    table(selected)
    numbers = sorted({f['line_no'] for f in selected})
    for n in (numbers[start:] if limit is None else numbers[start:start+limit]): show_ad(n)

print('Python:',platform.python_version(),'| файл:',TRAIN_PATH)
table([{'mcId':k,'Название':v} for k,v in CATEGORIES.items()],limit=None)
print('Исходная категория 101 в дополнительных метках запрещена. [] допустим.')


Python: 3.14.10 | файл: /home/Prakser/Projects/Innoglobalhack/data/raw/public-train-annotated.jsonl


mcId,Название
102,Штукатурные работы
103,Плиточные работы
104,Сантехника
105,Электрика
106,Поклейка обоев и малярные работы


Показано 5 из 5 строк таблицы.
Исходная категория 101 в дополнительных метках запрещена. [] допустим.


## 2. Проверяем каждую физическую строку JSONL

Ищем пустые строки, сломанную UTF-8/JSON, повтор ключей JSON и значения, которые не являются объектами.
Ошибочные строки показываем отдельно, **не превращаем в пустые корректные объявления**.
SHA-256 фиксирует версию файла. Число 10 000 сверяем с README, но читаем всё и при другом размере.


In [2]:
def strict_object(pairs):
    result = {}
    for key,value in pairs:
        if key in result: raise ValueError(f'Повтор JSON-ключа: {key!r}')
        result[key] = value
    return result
def reject_constant(value): raise ValueError(f'Недопустимая JSON-константа: {value}')
by_line,parse_issues = {},[]
digest = hashlib.sha256()
physical_lines = 0
with TRAIN_PATH.open('rb') as stream:
    for physical_lines,raw in enumerate(stream,1):
        digest.update(raw)
        try:
            text = raw.decode('utf-8')
            if not text.strip(): raise ValueError('Пустая физическая строка')
            obj = json.loads(text,object_pairs_hook=strict_object,parse_constant=reject_constant)
            if not isinstance(obj,dict): raise ValueError('Верхний уровень JSON должен быть объектом')
            by_line[physical_lines] = obj
        except (UnicodeDecodeError,ValueError,RecursionError) as exc:
            parse_issues.append({'line_no':physical_lines,'reason':str(exc),
                'raw_preview':raw[:250].decode('utf-8',errors='replace')})
SOURCE_SHA256 = digest.hexdigest()
remember('file_info',[{'file':TRAIN_PATH.name,'bytes':TRAIN_PATH.stat().st_size,
    'physical_lines':physical_lines,'parsed_objects':len(by_line),'parse_failures':len(parse_issues),
    'sha256':SOURCE_SHA256}],limit=None)
register('parse','Читаемость всех строк JSONL','ОШИБКА',parse_issues,physical_lines,
    'Каждая строка — корректный UTF-8 JSON-объект без повторных ключей.',
    'Восстановить строки из источника; ниже они не участвуют в статистике объявлений.')
print('Размер совпал с README.' if physical_lines==EXPECTED_ROWS else
      f'ВНИМАНИЕ: в README {EXPECTED_ROWS}, в файле {physical_lines} строк. Проверь версию.')


Таблица: file_info


file,bytes,physical_lines,parsed_objects,parse_failures,sha256
public-train-annotated.jsonl,46350817,10000,10000,0,b7118d68086624fbeab43839f22a635d4e325e4ee339f22d4aa0ad3c2e986615


Показано 1 из 1 строк таблицы.

Читаемость всех строк JSONL [parse] — ОШИБКА
Проверено: 10,000; строк со срабатыванием: 0; событий: 0.
Правило: Каждая строка — корректный UTF-8 JSON-объект без повторных ключей. 
Действие: Восстановить строки из источника; ниже они не участвуют в статистике объявлений.


Таблица / полные объявления (HTML)

Совпадений нет.
Размер совпал с README.


## 3. Проверяем поля, типы и каждую метку

Ожидаем item_id: int, непустые title/description и массив found_microcategories.
В массиве — объекты с mcId: int и label: str, ID 102–106 без повторов и с точным названием из справочника.
True не считаем целочисленным ID. Лишние поля отмечаем.
Невалидные записи исключаем **только из следующих расчётов**, сохраняя номера и причины. Исходник не меняется.


In [3]:
required = {'item_id','title','description','found_microcategories'}
schema_issues,extra_fields = [],[]
field_types = defaultdict(Counter)
for n,row in by_line.items():
    errors = []
    for field in required:
        field_types[field][type(row[field]).__name__ if field in row else 'MISSING'] += 1
    missing = required-row.keys()
    if missing: errors.append('Нет полей: '+', '.join(sorted(missing)))
    if type(row.get('item_id')) is not int: errors.append('item_id не int')
    for field in ['title','description']:
        value = row.get(field)
        if not isinstance(value,str): errors.append(f'{field} не строка')
        elif not value.strip(): errors.append(f'{field} пустое/из пробелов')
    labels = row.get('found_microcategories')
    if not isinstance(labels,list): errors.append('found_microcategories не массив')
    else:
        seen = set()
        for j,label in enumerate(labels):
            if not isinstance(label,dict):
                errors.append(f'Метка [{j}] не объект'); continue
            mc = label.get('mcId')
            if type(mc) is not int or mc not in CATEGORIES:
                errors.append(f'Метка [{j}]: запрещённый mcId={mc!r}')
            else:
                if mc in seen: errors.append(f'Повтор mcId={mc}')
                seen.add(mc)
                if label.get('label')!=CATEGORIES[mc]:
                    errors.append(f'Для {mc} неверное название: {label.get("label")!r}')
            if set(label)!={'mcId','label'}: errors.append(f'Метка [{j}]: поля отличаются от mcId/label')
    if errors: schema_issues.append({'line_no':n,'reason':'; '.join(errors)})
    if row.keys()-required: extra_fields.append({'line_no':n,'reason':', '.join(sorted(row.keys()-required))})
bad_schema = {e['line_no'] for e in schema_issues}
train = [dict(row,line_no=n,labels=tuple(sorted(x['mcId'] for x in row['found_microcategories'])))
         for n,row in by_line.items() if n not in bad_schema]
N = len(train)
remember('field_types',[{'field':f,'type':t,'rows':count}
    for f,types in sorted(field_types.items()) for t,count in sorted(types.items())],limit=None)
register('schema','Соответствие структуре и справочнику','ОШИБКА',schema_issues,len(by_line),
    'Типы полей, непустой текст, допустимые уникальные ID и точные названия.',
    'Разобрать причины; не исправлять смысл меток простой заменой ID.')
register('extra_fields','Дополнительные поля','ИНФОРМАЦИЯ',extra_fields,len(by_line),
    'Поля верхнего уровня вне четырёх ожидаемых.','Проверить, не изменилась ли схема поставки.')
print(f'Физических строк: {physical_lines}; корректных: {N}; ошибки чтения: {len(parse_issues)}; схемы: {len(bad_schema)}.')
assert physical_lines==len(by_line)+len(parse_issues)
assert len(by_line)==N+len(bad_schema)
if not train: raise RuntimeError('Нет корректных записей. Сначала исправь чтение/схему.')
show_ad(train[0]['line_no'])


Таблица: field_types


field,type,rows
description,str,10000
found_microcategories,list,10000
item_id,int,10000
title,str,10000


Показано 4 из 4 строк таблицы.

Соответствие структуре и справочнику [schema] — ОШИБКА
Проверено: 10,000; строк со срабатыванием: 0; событий: 0.
Правило: Типы полей, непустой текст, допустимые уникальные ID и точные названия. 
Действие: Разобрать причины; не исправлять смысл меток простой заменой ID.


Таблица / полные объявления (HTML)

Совпадений нет.

Дополнительные поля [extra_fields] — ИНФОРМАЦИЯ
Проверено: 10,000; строк со срабатыванием: 0; событий: 0.
Правило: Поля верхнего уровня вне четырёх ожидаемых. 
Действие: Проверить, не изменилась ли схема поставки.


Таблица / полные объявления (HTML)

Совпадений нет.
Физических строк: 10000; корректных: 10000; ошибки чтения: 0; схемы: 0.


Таблица / полные объявления (HTML)

## 4. Идентификаторы

Повтор item_id может быть повтором выгрузки либо разными версиями объявления.
Если тексты/метки различаются, правильную версию нужно уточнить.
Неположительный ID — подозрение, не нарушение заданного диапазона: README диапазон не определяет.


In [4]:
id_groups = defaultdict(list)
for r in train: id_groups[r['item_id']].append(r)
id_events = []
for item_id,group in id_groups.items():
    if len(group)<2: continue
    versions = len({(r['title'],r['description'],r['labels']) for r in group})
    id_events.extend({'line_no':r['line_no'],'item_id':item_id,'group_size':len(group),
                      'reason':f'Разных вариантов текста/меток: {versions}'} for r in group)
register('duplicate_id','Повторяющиеся item_id','ПРОВЕРИТЬ',id_events,N,
    'Один item_id в нескольких строках.','Сверить версии и источник ID.')
register('unusual_id','Подозрительные значения item_id','ПРОВЕРИТЬ',
    [{'line_no':r['line_no'],'reason':f"item_id={r['item_id']}"} for r in train if r['item_id']<=0],N,
    'item_id <= 0, эвристика.','Уточнить правила идентификаторов.')



Повторяющиеся item_id [duplicate_id] — ПРОВЕРИТЬ
Проверено: 10,000; строк со срабатыванием: 0; событий: 0.
Правило: Один item_id в нескольких строках. 
Действие: Сверить версии и источник ID.


Таблица / полные объявления (HTML)

Совпадений нет.

Подозрительные значения item_id [unusual_id] — ПРОВЕРИТЬ
Проверено: 10,000; строк со срабатыванием: 0; событий: 0.
Правило: item_id <= 0, эвристика. 
Действие: Уточнить правила идентификаторов.


Таблица / полные объявления (HTML)

Совпадений нет.


## 5. Длины и информативность текста

Считаем символы и слова для всех записей. Квантили показывают типичный текст и длинный хвост.
Порог **40 символов** в описании — повод посмотреть, а не удалить: услуга может быть в заголовке.
Число символов **не равно** числу токенов модели.


In [5]:
def quantile(values,q):
    values = sorted(values)
    pos = (len(values)-1)*q
    lo,hi = math.floor(pos),math.ceil(pos)
    return values[lo]+(values[hi]-values[lo])*(pos-lo)
for r in train:
    r['text'] = r['title']+'\n'+r['description']
    r['title_chars'],r['description_chars'] = len(r['title']),len(r['description'])
    r['words'] = len(re.findall(r'\w+',r['text']))
lengths = []
for field in ['title_chars','description_chars','words']:
    values = [r[field] for r in train]
    lengths.append({'field':field,'min':min(values),'mean':round(statistics.mean(values),1),
        **{f'p{q*100:g}':round(quantile(values,q),1) for q in [.01,.25,.5,.75,.95,.99]},'max':max(values)})
remember('lengths',lengths,limit=None)
edges = [(0,40),(40,200),(200,500),(500,1000),(1000,2000),(2000,4000),(4000,float('inf'))]
bars('Длины описаний, символы',[(f'{lo}–{hi if math.isfinite(hi) else "∞"}',
    sum(lo<=r['description_chars']<hi for r in train)) for lo,hi in edges])
register('short_text','Очень короткие описания','ПРОВЕРИТЬ',
    [{'line_no':r['line_no'],'reason':f"Описание: {r['description_chars']} символов",'labels':r['labels']}
     for r in train if r['description_chars']<40],N,
    'Описание короче 40 символов.','Прочитать вместе с заголовком.')
register('no_letters','Текст без букв','ПРОВЕРИТЬ',
    [{'line_no':r['line_no'],'reason':'Нет букв Unicode'} for r in train if not any(c.isalpha() for c in r['text'])],N,
    'Ни одного буквенного символа в title+description.','Проверить пригодность входа для обучения.')
remember('longest',[{'line_no':r['line_no'],'title':r['title'],'description_chars':r['description_chars'],'labels':r['labels']}
    for r in sorted(train,key=lambda x:x['description_chars'],reverse=True)[:10]],limit=None)


Таблица: lengths


field,min,mean,p1,p25,p50,p75,p95,p99,max
title_chars,6,39.4,14.0,33.0,42.0,47.0,50.0,57.0,88
description_chars,1,2324.0,20.0,1384.8,2196.0,3214.0,4960.0,5438.0,5564
words,3,309.2,7.0,184.0,300.0,427.0,638.0,730.0,789


Показано 3 из 3 строк таблицы.


Таблица / полные объявления (HTML)


Очень короткие описания [short_text] — ПРОВЕРИТЬ
Проверено: 10,000; строк со срабатыванием: 236; событий: 236.
Правило: Описание короче 40 символов. 
Действие: Прочитать вместе с заголовком.


line_no,reason,labels
48,Описание: 22 символов,()
55,Описание: 26 символов,()
70,Описание: 26 символов,()
101,Описание: 4 символов,()
145,Описание: 15 символов,()
153,Описание: 21 символов,"(103,)"
197,Описание: 32 символов,()
276,Описание: 32 символов,()
400,Описание: 6 символов,()
406,Описание: 7 символов,()


... (4 outputs left out)

Таблица / полные объявления (HTML)

Совпадений нет.
Таблица: longest


line_no,title,description_chars,labels
4776,Самостоятельный ремонт ипотечных квартир Lime Wash,5564,"(106,)"
2555,Сделаем ремонт вашей квартиры без шабашников в Уфе,5536,"(103, 104, 105)"
6849,Ремонт под ключ — от дизайна до готовой квартиры,5520,"(102, 103, 104, 105, 106)"
3361,"Ремонт квартир, коттеджей, домов",5488,"(102, 103, 104, 105, 106)"
7356,"Ремонт новостройки, вторичного жилья 44 дня",5483,"(102, 103, 104, 105, 106)"
5778,Ремонт квартир под ключ. Смета и договор,5458,()
8881,Ремонт квартиры под ключ Цена,5457,"(103, 104, 105, 106)"
602,Ремонт квартир под ключ в Москве мо,5456,"(102, 103, 104, 105, 106)"
6505,Ремонт однокомнатной квартиры под ключ,5456,"(104, 105)"
8330,Ремонт квартир и домов под ключ,5454,"(103, 104, 105)"


Показано 10 из 10 строк таблицы.


## 6. Метки, все сочетания и совместная встречаемость

Показываем пять категорий, **все 32 возможных набора**, в том числе отсутствующие, и число меток 0–5.
Матрица совместной встречаемости считает объявления с обеими категориями; диагональ — частота самой категории.
Редкие сочетания стоит включить в ручную проверку. Частота сама по себе не доказывает ошибку.


In [6]:
label_counts = Counter(mc for r in train for mc in r['labels'])
remember('category_frequency',[{'mcId':mc,'category':name,'rows':label_counts[mc],
    'percent':round(100*label_counts[mc]/N,2)} for mc,name in CATEGORIES.items()],limit=None)
bars('Объявления с меткой',[(f'{mc}: {name}',label_counts[mc]) for mc,name in CATEGORIES.items()])
cardinality = Counter(len(r['labels']) for r in train)
bars('Количество дополнительных категорий',[(str(k),cardinality[k]) for k in range(6)])
combinations = Counter(r['labels'] for r in train)
all_combinations = [c for k in range(6) for c in itertools.combinations(CATEGORIES,k)]
remember('label_combinations',[{'labels':c or '[]','rows':combinations[c],'percent':round(100*combinations[c]/N,2)}
    for c in sorted(all_combinations,key=lambda c:(-combinations[c],c))],limit=None)
remember('cooccurrence',[{'mcId':a,**{str(b):sum(a in r['labels'] and b in r['labels'] for r in train)
    for b in CATEGORIES}} for a in CATEGORIES],limit=None)
print(f'Без меток: {cardinality[0]:,} ({cardinality[0]/N:.1%}); все пять: {cardinality[5]:,} ({cardinality[5]/N:.1%}); '
      f'среднее: {sum(label_counts.values())/N:.2f}.')


Таблица: category_frequency


mcId,category,rows,percent
102,Штукатурные работы,5484,54.84
103,Плиточные работы,5073,50.73
104,Сантехника,4582,45.82
105,Электрика,4562,45.62
106,Поклейка обоев и малярные работы,5481,54.81


Показано 5 из 5 строк таблицы.


Таблица / полные объявления (HTML)

Таблица / полные объявления (HTML)

Таблица: label_combinations


labels,rows,percent
"(102, 103, 104, 105, 106)",3173,31.73
[],2909,29.09
"(102, 103, 106)",735,7.35
"(102, 106)",553,5.53
"(106,)",511,5.11
"(104, 105)",294,2.94
"(103, 104, 105)",261,2.61
"(102,)",223,2.23
"(102, 103, 104, 105)",187,1.87
"(102, 104, 105)",156,1.56


Показано 32 из 32 строк таблицы.
Таблица: cooccurrence


mcId,102,103,104,105,106
102,5484,4411,3730,3776,4744
103,4411,5073,3944,3897,4224
104,3730,3944,4582,4320,3496
105,3776,3897,4320,4562,3586
106,4744,4224,3496,3586,5481


Показано 5 из 5 строк таблицы.
Без меток: 2,909 (29.1%); все пять: 3,173 (31.7%); среднее: 2.52.


## 7. Дубликаты и противоречия

Четыре определения: **точный заголовок+описание**, **только описание**,
**нормализованный заголовок+описание**, **нормализованное описание**.
Нормализация: Unicode NFKC, регистр, ё/е, пробелы и невидимые форматирующие символы.
Разные метки у точного входа — противоречие. После нормализации — кандидат на проверку.
Одинаковые описания с разными заголовками не обязательно противоречат друг другу.

«Лишние повторы» = размер каждой группы минус один. Это отличается от всех строк в группах.
Исходники сохраняются. Семантические перефразы таким сравнением **не обнаруживаются**.


In [7]:
def normalize_key(text):
    text = unicodedata.normalize('NFKC',text).casefold().replace('ё','е')
    return ' '.join(''.join(c for c in text if unicodedata.category(c)!='Cf').split())
def key_hash(key):
    return hashlib.sha256(json.dumps(key,ensure_ascii=False).encode('utf-8')).hexdigest()
groupings = {name:defaultdict(list) for name in
             ['exact_input','description','normalized_input','normalized_description']}
for r in train:
    keys = {'exact_input':(r['title'],r['description']),'description':r['description'],
        'normalized_input':(normalize_key(r['title']),normalize_key(r['description'])),
        'normalized_description':normalize_key(r['description'])}
    for name,key in keys.items(): groupings[name][key].append(r)
    r['duplicate_group'] = key_hash(keys['exact_input'])
    r['normalized_description_group'] = key_hash(keys['normalized_description'])
duplicate_stats,duplicate_members,duplicate_events,conflict_events,normalized_conflicts = [],[],[],[],[]
for name,groups in groupings.items():
    repeated = [g for g in groups.values() if len(g)>1]
    conflicts = [g for g in repeated if len({r['labels'] for r in g})>1]
    duplicate_stats.append({'definition':name,'unique_groups':len(groups),'duplicate_groups':len(repeated),
        'rows_in_groups':sum(map(len,repeated)),'excess_rows':sum(len(g)-1 for g in repeated),'conflict_groups':len(conflicts)})
    for key,group in groups.items():
        if len(group)<2: continue
        conflict = len({r['labels'] for r in group})>1
        for r in group:
            event = {'line_no':r['line_no'],'group':key_hash(key),'group_size':len(group),
                     'labels':r['labels'],'reason':f'{name}; разные метки={conflict}'}
            duplicate_members.append({'definition':name,**event})
            if name=='exact_input':
                duplicate_events.append(event)
                if conflict: conflict_events.append(event)
            if name=='normalized_input' and conflict: normalized_conflicts.append(event)
remember('duplicate_summary',duplicate_stats,limit=None)
tables['duplicate_members'] = duplicate_members
register('exact_duplicates','Полностью одинаковые входные тексты','РИСК УТЕЧКИ',duplicate_events,N,
    'Совпадают title и description.','Всю группу оставлять в одной части train/validation.')
register('exact_conflicts','Противоречивые метки точных дубликатов','ОШИБКА',conflict_events,N,
    'Одинаковые title/description, разные наборы mcId.','Согласовать разметку всей группы вручную.')
register('normalized_conflicts','Противоречия после нормализации','ПРОВЕРИТЬ',normalized_conflicts,N,
    'Нормализованные title/description одинаковы, метки разные.','Сначала сравнить исходники.')
unique_exact = [g[0] for g in groupings['exact_input'].values()]
remember('frequency_without_duplicates',[{'mcId':mc,'all_rows_percent':round(100*label_counts[mc]/N,2),
    'one_per_exact_input_percent':round(100*sum(mc in r['labels'] for r in unique_exact)/len(unique_exact),2)}
    for mc in CATEGORIES],limit=None)
print('Доли после схлопывания — диагностика. При конфликте взят первый вариант; это не исправленный датасет.')


Таблица: duplicate_summary


definition,unique_groups,duplicate_groups,rows_in_groups,excess_rows,conflict_groups
exact_input,5002,402,5400,4998,0
description,4866,413,5547,5134,0
normalized_input,4996,401,5405,5004,0
normalized_description,4854,413,5559,5146,0


Показано 4 из 4 строк таблицы.

Полностью одинаковые входные тексты [exact_duplicates] — РИСК УТЕЧКИ
Проверено: 10,000; строк со срабатыванием: 5,400; событий: 5,400.
Правило: Совпадают title и description. 
Действие: Всю группу оставлять в одной части train/validation.


line_no,group,group_size,labels,reason
3,c49f212d0e50026e8395ba14e5652208fdf4310fbcc5697ce7a38e2779a01b4f,48,"(102, 103, 104, 105, 106)",exact_input; разные метки=False
43,c49f212d0e50026e8395ba14e5652208fdf4310fbcc5697ce7a38e2779a01b4f,48,"(102, 103, 104, 105, 106)",exact_input; разные метки=False
214,c49f212d0e50026e8395ba14e5652208fdf4310fbcc5697ce7a38e2779a01b4f,48,"(102, 103, 104, 105, 106)",exact_input; разные метки=False
427,c49f212d0e50026e8395ba14e5652208fdf4310fbcc5697ce7a38e2779a01b4f,48,"(102, 103, 104, 105, 106)",exact_input; разные метки=False
483,c49f212d0e50026e8395ba14e5652208fdf4310fbcc5697ce7a38e2779a01b4f,48,"(102, 103, 104, 105, 106)",exact_input; разные метки=False
501,c49f212d0e50026e8395ba14e5652208fdf4310fbcc5697ce7a38e2779a01b4f,48,"(102, 103, 104, 105, 106)",exact_input; разные метки=False
523,c49f212d0e50026e8395ba14e5652208fdf4310fbcc5697ce7a38e2779a01b4f,48,"(102, 103, 104, 105, 106)",exact_input; разные метки=False
666,c49f212d0e50026e8395ba14e5652208fdf4310fbcc5697ce7a38e2779a01b4f,48,"(102, 103, 104, 105, 106)",exact_input; разные метки=False
862,c49f212d0e50026e8395ba14e5652208fdf4310fbcc5697ce7a38e2779a01b4f,48,"(102, 103, 104, 105, 106)",exact_input; разные метки=False
867,c49f212d0e50026e8395ba14e5652208fdf4310fbcc5697ce7a38e2779a01b4f,48,"(102, 103, 104, 105, 106)",exact_input; разные метки=False


Показано 15 из 5400 строк таблицы.


Таблица / полные объявления (HTML)

... (4 outputs left out)

Таблица / полные объявления (HTML)

Совпадений нет.
Таблица: frequency_without_duplicates


mcId,all_rows_percent,one_per_exact_input_percent
102,54.84,51.18
103,50.73,48.92
104,45.82,47.98
105,45.62,47.64
106,54.81,54.76


Показано 5 из 5 строк таблицы.
Доли после схлопывания — диагностика. При конфликте взят первый вариант; это не исправленный датасет.


## 8. Артефакты текста

Проверяем заголовок и описание. Каждое правило выводит число строк, фрагмент и полные примеры.
Эмодзи/контакты — информационные признаки. Смешанные алфавиты, складские приписки,
невидимые символы и похожие на инструкции фразы — повод проверить контекст.
Регулярные выражения допускают ложные срабатывания; найденная фраза не доказывает намерение автора.


In [8]:
ARTIFACT_RULES = {
 'mixed_alphabet':('Слова со смешанной латиницей/кириллицей','ПРОВЕРИТЬ',r'(?=[A-Za-zА-Яа-яЁё]*[A-Za-z])(?=[A-Za-zА-Яа-яЁё]*[А-Яа-яЁё])[A-Za-zА-Яа-яЁё]+'),
 'replacement_character':('Символ повреждённой кодировки','ПРОВЕРИТЬ','\ufffd'),
 'html_tags':('HTML-теги','ПРОВЕРИТЬ',r'</?[a-z][^>]{0,200}>'),
 'escaped_newline':('Буквальные \\n вместо переноса','ПРОВЕРИТЬ',r'\\[nr]'),
 'stock_noise':('Складские/товарные приписки','ПРОВЕРИТЬ',r'место на складе|номер продаваемого оборудования|размеры оборудования на фото'),
 'seo_marker':('Явные поисковые приписки','ПРОВЕРИТЬ',r'для поиска|ключевые слова|поисковые запросы|нас (?:также )?ищут'),
 'prompt_like':('Похожие на инструкции модели фразы','ПРОВЕРИТЬ',r'игнорируй.{0,40}инструкц|ignore.{0,40}instructions|system\s*prompt|ответь.{0,30}json|ты.{0,15}(?:chatgpt|ассистент)'),
 'emoji':('Эмодзи/декоративные символы','ИНФОРМАЦИЯ',r'[\U0001F300-\U0001FAFF\u2600-\u27BF]'),
 'link':('Ссылки','ИНФОРМАЦИЯ',r'https?://[^\s]+|www\.[^\s]+|t\.me/[^\s]+'),
 'phone':('Похожие на телефоны последовательности','ИНФОРМАЦИЯ',r'(?:\+7|\b8)[\s(-]*\d{3}[\s)-]*\d{3}[\s-]*\d{2}[\s-]*\d{2}'),
}
def snippet(text,start,end,width=100):
    return text[max(0,start-width):start]+'[['+text[start:end]+']]'+text[end:end+width]
for key,(title,level,pattern) in ARTIFACT_RULES.items():
    regex = re.compile(pattern,re.I)
    events = []
    for r in train:
        matches = list(regex.finditer(r['text']))
        if matches:
            m = matches[0]
            events.append({'line_no':r['line_no'],'occurrences':len(matches),'reason':snippet(r['text'],m.start(),m.end())})
    register(key,title,level,events,N,pattern,'Посмотреть контекст; исходник и метки автоматически не менять.')
invisible = []
for r in train:
    chars = Counter(f'U+{ord(c):04X} {unicodedata.name(c,"UNKNOWN")}' for c in r['text']
        if unicodedata.category(c)=='Cf' or (unicodedata.category(c)=='Cc' and c not in '\n\r\t'))
    if chars: invisible.append({'line_no':r['line_no'],'reason':dict(chars)})
register('invisible','Невидимые/управляющие символы','ПРОВЕРИТЬ',invisible,N,
    'Unicode Cf и Cc, кроме обычных переносов/табуляции.',
    'Проверить влияние на поиск и токенизацию; часть Cf используется в эмодзи.')



Слова со смешанной латиницей/кириллицей [mixed_alphabet] — ПРОВЕРИТЬ
Проверено: 10,000; строк со срабатыванием: 541; событий: 541.
Правило: (?=[A-Za-zА-Яа-яЁё]*[A-Za-z])(?=[A-Za-zА-Яа-яЁё]*[А-Яа-яЁё])[A-Za-zА-Яа-яЁё]+ 
Действие: Посмотреть контекст; исходник и метки автоматически не менять.


line_no,occurrences,reason
10,1,"месяца действуют скидки на материалы до 30%! Ремонт начинается с понятной сметы и точных сроков. В [[PROквадраты]] фиксируем стоимость, этапы и сроки в договоре. Даём гарантию на работы — 1 год. Работаем своей ком"
15,1,"заменить ламинат, укладка напольных покрытий, положить ламинат, постелить ламинат, монтаж ламината, [[мoнтаж]] плинтуса, ремонт квартир, ремонт квартиры, ремонт квартир под ключ, ремонт квартиры под ключ, компл"
44,27,Ремонт и строительство под ключ с гарантией 🏠 [[Планируeте]] cтpоительство или ремoнт? Нaпишите нaм — пoможeм вoплoтить ваш прoeкт в жизнь! Oт фундaмeнта до го
52,1,"ение перегородок из газоблока пеноблока пгп ВОЗВЕДЕНИЕ ПЕРЕГОРОДОК В МОСКВЕ И МО ГАРАНТИЯ 3 ГОДА 🏗 «[[АкимовPRO]]» — профессиональное возведение межкомнатных перегородок из газоблока, пеноблока и кирпича. Более 1"
63,7,"юбой сложности: хрущёвки, сталинки, лофты, коттеджи — даже если стены кривые как маршрут таксиста. [[Cразу]] скaжу - мелочёвкoй не зaнимaюсь! Tипа: 🛑 частичная замена плитки 🛑 замена напольного покрытия 🛑"
87,27,Ремонт квартир и домов под ключ Ремонт квартир и домов под ключ 🔥 [[РЕMOHT]] КВАРТИР ПOД КЛЮЧ OТ ПРОФЕCСИOНAЛOB БEЗ ПOCРЕДНИКOВ 🚨 Hапишитe прямо ceйчaс — отвeтим в течениe неc
95,2,"вартиры, услуги по ремонту квартир, ремонт и отделка квартир, ремонт квартир с нуля, ремонт ванной, [[рeмонт]] ванных кoмнaт, плиточник, ванна под ключ, туалет под ключ"
119,2,"т кухни, ремонт ванной комнаты, чистовая отделка квартиры, отремонтировать квартиру, ремонт ванной, [[рeмонт]] ванных кoмнaт, плиточник, ванна под ключ, туалет под ключ, уложить плитку, положить плитку, ремонт"
135,27,Ремонт квартир и домов под ключ 🏠 [[Рeмoнт]] квартир и домов под ключ — берём на себя вeсь пpоцeсc: от дeмoнтaжа и чepнoвыx pабот до чистoвой oт
142,12,"✅ Консультация, Замер, Смета - бесплатно ✅ Полноценный клининг после уборк Привет! Меня Василий, я [[прорaб]] с комaндой из 12 oпытных мaстеров. Среди нас инжeнеры, элeктрики, плитoчники, мaляры-штукaтуры, мaс"


Показано 15 из 541 строк таблицы.


Таблица / полные объявления (HTML)

Таблица / полные объявления (HTML)


Символ повреждённой кодировки [replacement_character] — ПРОВЕРИТЬ
Проверено: 10,000; строк со срабатыванием: 0; событий: 0.
Правило: � 
Действие: Посмотреть контекст; исходник и метки автоматически не менять.


... (27 outputs left out)

line_no,reason
16,{'U+200D ZERO WIDTH JOINER': 1}
17,{'U+200B ZERO WIDTH SPACE': 5}
18,{'U+200D ZERO WIDTH JOINER': 1}
29,{'U+200D ZERO WIDTH JOINER': 1}
89,{'U+200D ZERO WIDTH JOINER': 1}
95,{'U+200D ZERO WIDTH JOINER': 1}
97,{'U+200D ZERO WIDTH JOINER': 1}
117,{'U+200D ZERO WIDTH JOINER': 1}
213,{'U+200B ZERO WIDTH SPACE': 5}
225,{'U+200D ZERO WIDTH JOINER': 1}


Показано 15 из 540 строк таблицы.


Таблица / полные объявления (HTML)

Таблица / полные объявления (HTML)

## 9. Словарь услуг: каждая пара «строка × категория»

Это диагностический словарь, **не эталон**. Ищем услуги в заголовке и описании.
«Слово есть, метки нет» может быть отрицанием/неуслугой; «метка есть, слова нет» — неполным словарём либо ошибкой разметки.
Показываем сами правила, все четыре исхода и F1 согласия. Высокое согласие не доказывает,
как получена разметка, и не измеряет качество на скрытом тесте.


In [9]:
KEYWORDS = {
102:r'\b(?:штукатур\w*|оштукатур\w*)',
103:r'\b(?:плитк\w*|плиточ\w*|кафел\w*|керамогранит\w*|мозаик\w*|затир\w*\s+шв\w*)',
104:r'\b(?:сантех\w*|водоснаб\w*|водопровод\w*|канализац\w*|смесител\w*|унитаз\w*|раковин\w*|инсталляц\w*|радиатор\w*|полотенцесушител\w*|сифон\w*|засор\w*|протеч\w*|бойлер\w*|водонагревател\w*|разводк\w*\s+труб\w*)',
105:r'\b(?:электри\w*|электромонтаж\w*|электропровод\w*|проводк\w*|розет\w*|выключател\w*|электрощит\w*|люстр\w*|светильник\w*|кабел\w*|узо|дифавтомат\w*)',
106:r'\b(?:обои|обоев|обоям\w*|обоях|поклей\w*|оклей\w*|маляр\w*|покрас\w*|окрас\w*|шпакл\w*|шпатл\w*|стеклохолст\w*|грунтов\w*)',
}
remember('keyword_rules',[{'mcId':mc,'category':CATEGORIES[mc],'regex':p} for mc,p in KEYWORDS.items()],limit=None)
HOMOGLYPHS = str.maketrans('aceopxyABCEHKMOPTXY','асеорхуАВСЕНКМОРТХУ')
def search_text(text):
    # Замены только 1 символ → 1 символ: позиции относятся к исходнику.
    text = re.sub(r'\w+',lambda m:m.group().translate(HOMOGLYPHS)
        if re.search('[а-яёА-ЯЁ]',m.group()) and re.search('[a-zA-Z]',m.group()) else m.group(),text)
    return ''.join((c.lower() if len(c.lower())==1 else c).replace('ё','е') for c in text)
compiled = {mc:re.compile(p) for mc,p in KEYWORDS.items()}
keyword_without_label,label_without_keyword,title_only = [],[],[]
for r in train:
    r['search_text'] = search_text(r['text'])
    assert len(r['search_text'])==len(r['text'])
    r['hits'] = {mc:[(m.start(),m.end()) for m in regex.finditer(r['search_text'])] for mc,regex in compiled.items()}
    for mc,hits in r['hits'].items():
        base = {'line_no':r['line_no'],'mcId':mc,'current_label':int(mc in r['labels'])}
        if hits and mc not in r['labels']: keyword_without_label.append({**base,'reason':snippet(r['text'],*hits[0])})
        if not hits and mc in r['labels']:
            label_without_keyword.append({**base,'reason':'Метка есть; словарь не нашёл явного слова'})
        if hits and all(end<=len(r['title']) for start,end in hits):
            title_only.append({**base,'reason':snippet(r['text'],*hits[0])})
agreement = []
for mc in CATEGORIES:
    counts = Counter((mc in r['labels'],bool(r['hits'][mc])) for r in train)
    tp,fp,fn,tn = counts[True,True],counts[False,True],counts[True,False],counts[False,False]
    agreement.append({'mcId':mc,'слово+метка':tp,'слово без метки':fp,'метка без слова':fn,'нет обоих':tn,
        'F1 согласия':round(2*tp/(2*tp+fp+fn),4) if 2*tp+fp+fn else None})
remember('keyword_agreement',agreement,limit=None)
tp,fp,fn = (sum(x[k] for x in agreement) for k in ['слово+метка','слово без метки','метка без слова'])
print('micro-F1 согласия с авторазметкой:',round(2*tp/(2*tp+fp+fn),4) if 2*tp+fp+fn else 'не определён')
register('keyword_without_label','Слово есть, метки нет','ПРОВЕРИТЬ',keyword_without_label,N*5,
    'Словарь сработал, метки нет.','Проверить, предлагает ли исполнитель услугу.')
register('label_without_keyword','Метка есть, явного слова нет','ПРОВЕРИТЬ',label_without_keyword,N*5,
    'Метка есть, словарь не сработал.','Проверить синонимы, описание услуги и возможную лишнюю метку.')
register('title_only','Упоминание только в заголовке','ИНФОРМАЦИЯ',title_only,N*5,
    'Все совпадения категории находятся в title.','Не терять заголовок при подготовке входа модели.')


Таблица: keyword_rules


mcId,category,regex
102,Штукатурные работы,\b(?:штукатур\w*|оштукатур\w*)
103,Плиточные работы,\b(?:плитк\w*|плиточ\w*|кафел\w*|керамогранит\w*|мозаик\w*|затир\w*\s+шв\w*)
104,Сантехника,\b(?:сантех\w*|водоснаб\w*|водопровод\w*|канализац\w*|смесител\w*|унитаз\w*|раковин\w*|инсталляц\w*|радиатор\w*|полотенцесушител\w*|сифон\w*|засор\w*|протеч\w*|бойлер\w*|водонагревател\w*|разводк\w*\s+труб\w*)
105,Электрика,\b(?:электри\w*|электромонтаж\w*|электропровод\w*|проводк\w*|розет\w*|выключател\w*|электрощит\w*|люстр\w*|светильник\w*|кабел\w*|узо|дифавтомат\w*)
106,Поклейка обоев и малярные работы,\b(?:обои|обоев|обоям\w*|обоях|поклей\w*|оклей\w*|маляр\w*|покрас\w*|окрас\w*|шпакл\w*|шпатл\w*|стеклохолст\w*|грунтов\w*)


Показано 5 из 5 строк таблицы.
Таблица: keyword_agreement


mcId,слово+метка,слово без метки,метка без слова,нет обоих,F1 согласия
102,4260,373,1224,4143,0.8421
103,5072,387,1,4540,0.9632
104,4538,430,44,4988,0.9504
105,4542,388,20,5050,0.957
106,5474,544,7,3975,0.9521


Показано 5 из 5 строк таблицы.
micro-F1 согласия с авторазметкой: 0.9332

Слово есть, метки нет [keyword_without_label] — ПРОВЕРИТЬ
Проверено: 50,000; строк со срабатыванием: 1,173; событий: 2,122.
Правило: Словарь сработал, метки нет. 
Действие: Проверить, предлагает ли исполнитель услугу.


line_no,mcId,current_label,reason
18,106,0,"одных коттеджах и банях. — 🧱 Высококачественные черновые и чистовые работы: ровная штукатурка стен, [[шпатлевка]] под обои и покраску. — 💎 Качественная отделка и ремонт в новостройках, а также капитальное обновлен"
38,102,0,",ремонт в новостройках,ремонт во вторичке,ремонт квартиры, евроремонт, отделка под ключ, шпаклевка, [[штукатурка]], капительный ремонт, косметический ремонт,отделочные работы, плиточник, укладка плитки,бригада отде"
38,103,0,"тделка под ключ, шпаклевка, штукатурка, капительный ремонт, косметический ремонт,отделочные работы, [[плиточник]], укладка плитки,бригада отделочников, бригада строителей, отделка, малярные работы, демонтаж, монта"
38,104,0,"тки,бригада отделочников, бригада строителей, отделка, малярные работы, демонтаж, монтаж, установка [[сантехники]]."
38,106,0,"ел под ключ,ремонт в новостройках,ремонт во вторичке,ремонт квартиры, евроремонт, отделка под ключ, [[шпаклевка]], штукатурка, капительный ремонт, косметический ремонт,отделочные работы, плиточник, укладка плитки,"
42,103,0,"ы сделаем бесплатно и без обязательств. Напишите в чат — отвечу в тот же день. Также ищут: укладка [[плитки]], мастер по ремонту, ремонт квартиры под ключ, ремонт квартиры под ключ Одинцово, отделка квартир, р"
44,106,0,детали вашего проекта. Также ищите нас по: Ремонт квартир. Ремонт дома. Ремонт ванны. Штукатурка. [[Шпаклевка]]. Водоснабжение. Отопление. Теплый пол.Электрик освещение
52,102,0,"за, чтобы вы не переплачивали за последующую отделку. Кривые перегородки — это дополнительные мешки [[штукатурки]], увеличенный расход материалов, потерянное время и лишние расходы. Мы уделяем особое внимание геоме"
55,103,0,"Укладка [[плитки]], керамагранита, компле Ответственность и порядок."
58,104,0,монтаж карнизов монтаж розеток в бетон штробление стен под проводку установка подрозетников монтаж [[разводки труб]] ремонт ванной комнаты ремонт туалета


... (9 outputs left out)

line_no,mcId,current_label,reason
55,103,0,"Укладка [[плитки]], керамагранита, компле Ответственность и порядок."
117,106,1,Реставрация [[покраска]] покрытие ванны акрилом 🛁 РЕСТАВРАЦИЯ ВАНН ЖИДКИМ АКРИЛОМ | ЧАСТНЫЙ МАСТЕР 💎 Зачем покупать новую в
274,103,0,"[[Плиточник]] отделочник ремонт ванных комнат Сделаю Работу любой сложности!!! Быстро, качественно, красиво!!!"
348,106,0,"Наклейка [[обоев]] 10 10. Площадь ремонта 49,9 м2, Количество человек в бригаде 3, Бригадир Павлова В.А., Машина Опель"
524,103,0,"[[Плиточные]] и отделочные работы По ценам уточняйте, пишите, не всегда возле телефона"
588,102,0,"[[Штукатурка]],плитка, обои, Ремонт пот ключ Выполняю работа любой сложности"
588,103,0,"Штукатурка,[[плитка]], обои, Ремонт пот ключ Выполняю работа любой сложности"
588,106,0,"Штукатурка,плитка, [[обои]], Ремонт пот ключ Выполняю работа любой сложности"
800,106,0,[[Малярные]] работы шпаклевка стен Чистовая отделка под ключ Выполняем полный комплекс внутренних чистовых работ
809,103,0,"Укладка [[плитки]] Ремонт квартир,ремонт загородных домов,строительство загородных домов"


Показано 15 из 72 строк таблицы.


Таблица / полные объявления (HTML)

Таблица / полные объявления (HTML)

## 10. Отрицания, чужие услуги и поисковые хвосты

Разделяем явные отрицания, упоминания партнёров и вероятные хвосты ключевых слов.
«По запросу» и «оплачивается отдельно» сами по себе **не отрицания**.
Короткое контекстное окно ограничено пунктуацией. Возможны ложные срабатывания и пропуски.
Для каждой пары показываем, затронуты все или часть упоминаний (all_mentions), и текущую метку.


In [10]:
NEG_BEFORE = re.compile(r'(?:\bне\s+(?:делаем|выполняем|занимаемся|оказываем|предоставляем|беремся)|\bкроме\b|\bза исключением\b|\bбез\b)[^.!?;\n]{0,45}$')
NEG_AFTER = re.compile(r'^[^.!?;\n]{0,30}\bне\s+(?:делаем|выполняем|занимаемся|оказываем|предоставляем|входит|включен\w*)')
THIRD_PARTY = re.compile(r'партнер\w*|субподряд\w*|друг(?:ой|ие|их)\s+(?:мастер\w*|специалист\w*|исполнител\w*)')
TAIL_MARKER = re.compile(r'для поиска|ключевые слова|поисковые запросы|нас (?:также )?ищут')
def negative_context(text,start,end):
    return bool(NEG_BEFORE.search(text[max(0,start-80):start]) or NEG_AFTER.search(text[end:end+65]))
def third_party_context(text,start,end):
    before = re.split(r'[.!?;\n]',text[max(0,start-70):start])[-1]
    after = re.split(r'[.!?;\n]',text[end:end+70])[0]
    return bool(THIRD_PARTY.search(before+' '+text[start:end]+' '+after))
def tail_start(text):
    found = TAIL_MARKER.search(text)
    if found: return found.start()
    stripped = text.rstrip()
    last = stripped.split('\n')[-1]
    chunks = [x.strip() for x in last.split(',') if x.strip()]
    if len(chunks)>=7 and statistics.mean(map(len,chunks))<=40 and last.count('.')<=1:
        return len(stripped)-len(last)
    return len(text)
negatives,third_parties,tail_only = [],[],[]
for r in train:
    body = r['search_text'][len(r['title'])+1:]
    r['tail_start'] = len(r['title'])+1+tail_start(body)
    for mc,hits in r['hits'].items():
        if not hits: continue
        base = {'line_no':r['line_no'],'mcId':mc,'current_label':int(mc in r['labels'])}
        for detector,target in [(negative_context,negatives),(third_party_context,third_parties)]:
            suspicious = [h for h in hits if detector(r['search_text'],*h)]
            if suspicious:
                target.append({**base,'all_mentions':len(suspicious)==len(hits),'reason':snippet(r['text'],*suspicious[0])})
        if all(start>=r['tail_start'] for start,end in hits):
            tail_only.append({**base,'reason':snippet(r['text'],*hits[0])})
register('negation_context','Упоминания рядом с отрицанием','ПРОВЕРИТЬ',negatives,N*5,
    'Отрицание до/после словарного совпадения.','Проверить отрицание именно услуги, особенно all_mentions=True и current_label=1.')
register('third_party_context','Возможные услуги другого исполнителя','ПРОВЕРИТЬ',third_parties,N*5,
    'Рядом с услугой названы партнёры/другие мастера.','Уточнить, кто предлагает услугу.')
register('tail_only','Услуга только в вероятном поисковом хвосте','ПРОВЕРИТЬ',tail_only,N*5,
    'Все совпадения после маркера хвоста или в последней строке со списком >=7 коротких фраз.',
    'Прочитать весь текст: это может быть перечень реальных услуг.')
remember('context_summary',[{'check':name,'pairs':len(events),'with_label':sum(e['current_label'] for e in events),
    'rows':len({e['line_no'] for e in events})} for name,events in
    [('negation',negatives),('third_party',third_parties),('tail_only',tail_only)]],limit=None)



Упоминания рядом с отрицанием [negation_context] — ПРОВЕРИТЬ
Проверено: 50,000; строк со срабатыванием: 213; событий: 410.
Правило: Отрицание до/после словарного совпадения. 
Действие: Проверить отрицание именно услуги, особенно all_mentions=True и current_label=1.


line_no,mcId,current_label,all_mentions,reason
167,102,1,False,клёвка стен под обои ремонт под ключ дача ремонт частного дома бригада отделочников без посредников [[штукатурные]] работы в доме отделка холла и прихожей замена полов на даче подготовка стен под краску отделка дома
167,104,0,True,лка холла и прихожей замена полов на даче подготовка стен под краску отделка дома без электрики без [[сантехники]] только стены потолок пол
167,105,0,False,ты в доме отделка холла и прихожей замена полов на даче подготовка стен под краску отделка дома без [[электрики]] без сантехники только стены потолок пол
190,102,1,False,"ры Демонтаж старой отделки ⭐ ПОЧЕМУ ВЫБИРАЮТ НАС Все работы — своей бригадой, без субподрядчиков: [[штукатуры]], плиточники, электрик и сантехник работают вместе, поэтому сроки не срываются. Стены и полы выводим"
190,103,1,False,"старой отделки ⭐ ПОЧЕМУ ВЫБИРАЮТ НАС Все работы — своей бригадой, без субподрядчиков: штукатуры, [[плиточники]], электрик и сантехник работают вместе, поэтому сроки не срываются. Стены и полы выводим по уровню,"
190,105,1,False,"лки ⭐ ПОЧЕМУ ВЫБИРАЮТ НАС Все работы — своей бригадой, без субподрядчиков: штукатуры, плиточники, [[электрик]] и сантехник работают вместе, поэтому сроки не срываются. Стены и полы выводим по уровню, плитку кла"
275,102,1,False,"ие Демонтаж старой отделки ⭐ ПОЧЕМУ ВЫБИРАЮТ НАС Все работы — своей бригадой, без субподрядчиков: [[штукатуры]], плиточники, электрик и сантехник работают вместе, поэтому сроки не срываются. Стены и полы выводим"
275,103,1,False,"старой отделки ⭐ ПОЧЕМУ ВЫБИРАЮТ НАС Все работы — своей бригадой, без субподрядчиков: штукатуры, [[плиточники]], электрик и сантехник работают вместе, поэтому сроки не срываются. Стены и полы выводим по уровню,"
275,105,1,False,"лки ⭐ ПОЧЕМУ ВЫБИРАЮТ НАС Все работы — своей бригадой, без субподрядчиков: штукатуры, плиточники, [[электрик]] и сантехник работают вместе, поэтому сроки не срываются. Стены и полы выводим по уровню, плитку кла"
380,106,1,False,"ТEНЫ: выpавнивание, пoдготовкa,штукатурка, стяжка шпатлёвкa,пpoфессиoнaльная пoклeйкa oбоев или бeз [[поклейки]] oбoев под пoкpaску. Kлею плитки на стен и на полу. Гарантирую идеальный потолок и стены строго в ог"


Показано 15 из 410 строк таблицы.


Таблица / полные объявления (HTML)

Таблица / полные объявления (HTML)


Возможные услуги другого исполнителя [third_party_context] — ПРОВЕРИТЬ
Проверено: 50,000; строк со срабатыванием: 74; событий: 262.
Правило: Рядом с услугой названы партнёры/другие мастера. 
Действие: Уточнить, кто предлагает услугу.


... (8 outputs left out)

Таблица / полные объявления (HTML)

Таблица: context_summary


check,pairs,with_label,rows
negation,410,248,213
third_party,262,243,74
tail_only,543,148,346


Показано 3 из 3 строк таблицы.


## 11. Неоднозначные услуги на границах категорий

Собираем выравнивание стен, откосы, декоративную штукатурку, тёплый пол и ремонт ванной/санузла под ключ.
Это не ошибки: водяной и электрический тёплый пол — разные услуги, а название комнаты не доказывает сантехнические работы.
Показываем найденные строки с исходными наборами меток.


In [11]:
BOUNDARIES = {'wall_leveling':r'выравнив\w*\s+стен\w*','slopes':r'откос\w*',
    'decorative_plaster':r'декоратив\w*\s+штукатур\w*','warm_floor':r'тепл\w*\s+пол\w*',
    'bathroom_turnkey':r'(?:ванн\w*|сануз\w*)[^.!?\n]{0,35}под ключ'}
boundary_events = []
for r in train:
    for name,pattern in BOUNDARIES.items():
        match = re.search(pattern,r['search_text'])
        if match:
            boundary_events.append({'line_no':r['line_no'],'topic':name,'labels':r['labels'],
                'reason':snippet(r['text'],match.start(),match.end())})
register('boundary_context','Неоднозначные формулировки услуг','ПРОВЕРИТЬ',boundary_events,N,
    'Формулировки на границах категорий из справочника.','Согласовать трактовку на реальных примерах.')
remember('boundary_summary',[{'topic':t,'rows':sum(e['topic']==t for e in boundary_events)} for t in BOUNDARIES],limit=None)



Неоднозначные формулировки услуг [boundary_context] — ПРОВЕРИТЬ
Проверено: 10,000; строк со срабатыванием: 5,623; событий: 9,069.
Правило: Формулировки на границах категорий из справочника. 
Действие: Согласовать трактовку на реальных примерах.


line_no,topic,labels,reason
1,wall_leveling,"(102, 103, 104, 105, 106)","ные работы • Возведение и демонтаж перегородок • Монтаж стен из гипсокартона, газоблока и кирпича • [[Выравнивание стен]] и потолков • Штукатурка и шпаклёвка • Покраска стен и потолков • Поклейка обоев • Монтаж декоративн"
1,slopes,"(102, 103, 104, 105, 106)","вых кабин, раковин и унитазов • Монтаж радиаторов и тёплого пола • Установка дверей • Монтаж окон и [[откосов]] • Утепление балконов и лоджий • Сборка и монтаж мебели • Уборка и вывоз строительного мусора Предо"
1,decorative_plaster,"(102, 103, 104, 105, 106)","вание стен и потолков • Штукатурка и шпаклёвка • Покраска стен и потолков • Поклейка обоев • Монтаж [[декоративной штукатурки]] • Укладка ламината, линолеума, паркета и кварцвинила • Укладка плитки и керамогранита • Затирка шво"
1,warm_floor,"(102, 103, 104, 105, 106)","доснабжения и канализации • Установка ванн, душевых кабин, раковин и унитазов • Монтаж радиаторов и [[тёплого пола]] • Установка дверей • Монтаж окон и откосов • Утепление балконов и лоджий • Сборка и монтаж мебели •"
2,slopes,"(102, 103, 106)","ната, линолеума, кварцвинила ✔ Декоративная штукатурка ✔ Обшивка стен вагонкой и панелями ✔ Отделка [[откосов]] и проёмов ✔ Установка напольных плинтусов 🏠 КАКИЕ ОБЪЕКТЫ РЕМОНТИРУЕМ? 🔹 Квартиры в новостройках"
2,decorative_plaster,"(102, 103, 106)","олков ✔ Поклейка обоев ✔ Укладка плитки и керамогранита ✔ Монтаж ламината, линолеума, кварцвинила ✔ [[Декоративная штукатурка]] ✔ Обшивка стен вагонкой и панелями ✔ Отделка откосов и проёмов ✔ Установка напольных плинтусов 🏠 К"
3,wall_leveling,"(102, 103, 104, 105, 106)","ить теплопотери и подготовить поверхности под последующую отделку. 💥 Черновые строительные работы: [[выравнивание стен]] и потолков, устройство стяжки полов, монтаж или перенос перегородок, подготовка помещений под финиш"
4,slopes,"(102, 103, 104, 105, 106)","Чистовую отделку Покраску, обои, укладку плитки и напольных покрытий, установку плинтусов, отделку [[откосов]] Завершающий монтаж Двери, розетки, выключатели, светильники и сантехнические приборы — по согласова"
6,bathroom_turnkey,"(103, 104)","Ремонт [[санузла под ключ]] Приветствую Вас, меня зовут Антон. Я мастер по ремонту санузла. Выполняю полный цикл работ: от демо"
7,wall_leveling,"(102, 103, 104, 105, 106)","е и чистовые материалы). - Косметический ремонт (обои, краска, шпаклёвка, декоративные покрытия). - [[Выравнивание стен]] и потолков гипсокартоном. - Штукатурка и шпаклёвка под отделку. - Отделка стен деревом (вагонка, им"


Показано 15 из 9069 строк таблицы.


Таблица / полные объявления (HTML)

Таблица / полные объявления (HTML)

Таблица: boundary_summary


topic,rows
wall_leveling,3269
slopes,2017
decorative_plaster,805
warm_floor,1408
bathroom_turnkey,1570


Показано 5 из 5 строк таблицы.


## 12. Что потеряется при обрезке текста

Для 512/1024/2048/4096 **символов полного входа** считаем размеченные пары,
у которых ни одно словарное совпадение полностью не помещается в префикс.
Это риск потери явных слов, не измерение полноты смысла. Для выбранной модели нужен отдельный замер токенизатором.


In [12]:
truncation_rows,truncation_events = [],[]
for budget in [512,1024,2048,4096]:
    lost = []
    for r in train:
        for mc,hits in r['hits'].items():
            if mc in r['labels'] and hits and all(end>budget for start,end in hits):
                lost.append({'line_no':r['line_no'],'mcId':mc,'budget_chars':budget,'reason':snippet(r['text'],*hits[0])})
    truncation_rows.append({'budget_chars':budget,'truncated_rows':sum(len(r['text'])>budget for r in train),
        'affected_rows':len({e['line_no'] for e in lost}),'labelled_pairs_without_complete_hit':len(lost)})
    truncation_events.extend(lost)
remember('truncation',truncation_rows,limit=None)
register('truncation_risk','Риск потери явной услуги при обрезке','РИСК ОБРЕЗКИ',truncation_events,N*5*4,
    'Категория размечена; все совпадения за лимитом либо пересекают его.',
    'Выбрать длину по токенизатору; при необходимости обрабатывать части текста.')


Таблица: truncation


budget_chars,truncated_rows,affected_rows,labelled_pairs_without_complete_hit
512,8912,5378,16975
1024,8242,2874,8523
2048,5508,636,1407
4096,1320,39,41


Показано 4 из 4 строк таблицы.

Риск потери явной услуги при обрезке [truncation_risk] — РИСК ОБРЕЗКИ
Проверено: 200,000; строк со срабатыванием: 5,378; событий: 26,946.
Правило: Категория размечена; все совпадения за лимитом либо пересекают его. 
Действие: Выбрать длину по токенизатору; при необходимости обрабатывать части текста.


line_no,mcId,budget_chars,reason
1,102,512,"таж перегородок • Монтаж стен из гипсокартона, газоблока и кирпича • Выравнивание стен и потолков • [[Штукатурка]] и шпаклёвка • Покраска стен и потолков • Поклейка обоев • Монтаж декоративной штукатурки • Укладка"
1,103,512,"оев • Монтаж декоративной штукатурки • Укладка ламината, линолеума, паркета и кварцвинила • Укладка [[плитки]] и керамогранита • Затирка швов и герметизация • Монтаж натяжных и подвесных потолков • Установка на"
1,104,512,"Замена и монтаж розеток, выключателей и светильников • Прокладка кабеля и установка электрощитов • [[Сантехнические]] работы • Разводка водоснабжения и канализации • Установка ванн, душевых кабин, раковин и унитазов •"
1,105,512,"герметизация • Монтаж натяжных и подвесных потолков • Установка напольных и потолочных плинтусов • [[Электромонтажные]] работы • Замена и монтаж розеток, выключателей и светильников • Прокладка кабеля и установка электр"
1,106,512,"ок • Монтаж стен из гипсокартона, газоблока и кирпича • Выравнивание стен и потолков • Штукатурка и [[шпаклёвка]] • Покраска стен и потолков • Поклейка обоев • Монтаж декоративной штукатурки • Укладка ламината, ли"
2,102,512,"ТНО ПРИЕДЕМ, ОСМОТРИМ ПОМЕЩЕНИЕ И РАССЧИТАЕМ СТОИМОСТЬ РЕМОНТА! 🔧 ПОДГОТОВКА И ЧЕРНОВАЯ ОТДЕЛКА ✔ [[Штукатурка]] стен по маякам ✔ Шпаклёвка стен и потолков ✔ Выравнивание поверхностей под чистовую отделку ✔ Цемен"
2,103,512,"ой отделки 🎨 ФИНИШНЫЕ И ДЕКОРАТИВНЫЕ РАБОТЫ ✔ Покраска стен и потолков ✔ Поклейка обоев ✔ Укладка [[плитки]] и керамогранита ✔ Монтаж ламината, линолеума, кварцвинила ✔ Декоративная штукатурка ✔ Обшивка стен"
2,106,512,НИЕ И РАССЧИТАЕМ СТОИМОСТЬ РЕМОНТА! 🔧 ПОДГОТОВКА И ЧЕРНОВАЯ ОТДЕЛКА ✔ Штукатурка стен по маякам ✔ [[Шпаклёвка]] стен и потолков ✔ Выравнивание поверхностей под чистовую отделку ✔ Цементно-песчаная стяжка пола ✔
3,103,512,"строительные работы: шпаклёвка и окраска стен, оклейка обоями, укладка напольных покрытий (ламинат, [[плитка]], паркет), монтаж натяжных и подвесных потолков, установка межкомнатных дверей, плинтусов, наличнико"
3,104,512,"полная или частичная замена электропроводки, монтаж щитов и розеток, модернизация систем отопления, [[водоснабжения]] и канализации с установкой оборудования и запорной арматуры. 💥 Утепление стен, крыши и межэтажных"


Показано 15 из 26946 строк таблицы.


Таблица / полные объявления (HTML)

Таблица / полные объявления (HTML)

## 13. Утечка на демонстрационном train/validation split

Сравниваем воспроизводимый случайный split 80/20 с разбиением по группам нормализованных описаний.
Модель не обучаем. Группы формируем **без меток**, их размеры могут сдвинуть долю строк относительно 80/20.
Семантические перефразы всё ещё могут пересекаться. Отдельно контролируем item_id:
один ID с разными текстами требует объединения связанных групп перед настоящей валидацией.


In [13]:
rng = random.Random(RANDOM_SEED)
indices = list(range(N)); rng.shuffle(indices)
cut = max(1,int(N*.8))
random_train,random_val = [train[i] for i in indices[:cut]],[train[i] for i in indices[cut:]]
groups = sorted({r['normalized_description_group'] for r in train})
rng = random.Random(RANDOM_SEED); rng.shuffle(groups)
train_groups = set(groups[:max(1,int(len(groups)*.8))])
group_train = [r for r in train if r['normalized_description_group'] in train_groups]
group_val = [r for r in train if r['normalized_description_group'] not in train_groups]
def split_stats(name,left,right):
    exact = {r['duplicate_group'] for r in left}
    descriptions = {r['normalized_description_group'] for r in left}
    ids = {r['item_id'] for r in left}
    return {'split':name,'train_rows':len(left),'validation_rows':len(right),
        'validation_exact_overlap':sum(r['duplicate_group'] in exact for r in right),
        'validation_description_overlap':sum(r['normalized_description_group'] in descriptions for r in right),
        'validation_id_overlap':sum(r['item_id'] in ids for r in right)}
remember('split_leakage',[split_stats('random rows',random_train,random_val),
    split_stats('grouped descriptions',group_train,group_val)],limit=None)
remember('split_label_balance',[{'mcId':mc,
    'train_percent':round(100*sum(mc in r['labels'] for r in group_train)/len(group_train),2),
    'validation_percent':round(100*sum(mc in r['labels'] for r in group_val)/len(group_val),2) if group_val else None}
    for mc in CATEGORIES],limit=None)
assert not ({r['normalized_description_group'] for r in group_train}&{r['normalized_description_group'] for r in group_val})
print('Это пример группирования по выбранному ключу, не готовая утверждённая валидация.')


Таблица: split_leakage


split,train_rows,validation_rows,validation_exact_overlap,validation_description_overlap,validation_id_overlap
random rows,8000,2000,1094,1129,0
grouped descriptions,8218,1782,0,0,0


Показано 2 из 2 строк таблицы.
Таблица: split_label_balance


mcId,train_percent,validation_percent
102,54.27,57.46
103,50.5,51.8
104,45.42,47.64
105,45.25,47.31
106,54.76,55.05


Показано 5 из 5 строк таблицы.
Это пример группирования по выбранному ключу, не готовая утверждённая валидация.


## 14. Полная очередь ручной проверки и случайный контроль

В очереди **все** срабатывания «ОШИБКА»/«ПРОВЕРИТЬ», без усечения. У объявления может быть несколько причин.
Для чтения также создаём компактную очередь по точным текстам с номерами всех затронутых строк.
Приоритет задаёт порядок просмотра, не вероятность ошибки.

Отдельно берём до трёх случайных уникальных текстов на каждый встречающийся набор меток, включая строки без подозрений.
Это диагностическая, **не репрезентативная по частотам** выборка. Долю ошибок в ней нельзя переносить на весь train без взвешивания.


In [14]:
priority = {'parse':0,'schema':0,'exact_conflicts':0,'duplicate_id':1,'normalized_conflicts':1,
    'negation_context':2,'third_party_context':2,'stock_noise':2,
    'label_without_keyword':3,'keyword_without_label':3,'tail_only':3,'boundary_context':4}
review_events = [{**e,'priority':priority.get(e['check'],5),'decision':'','correct_mcIds':'','comment':'','reviewer':''}
    for e in findings if e['level'] in {'ОШИБКА','ПРОВЕРИТЬ'}]
review_events.sort(key=lambda e:(e['priority'],e['line_no'],e['check']))
events_by_line,review_groups = defaultdict(list),defaultdict(list)
for e in review_events: events_by_line[e['line_no']].append(e)
valid_by_line = {r['line_no']:r for r in train}
for n in events_by_line:
    group = valid_by_line[n]['duplicate_group'] if n in valid_by_line else f'invalid:{n}'
    review_groups[group].append(n)
compact_review = []
for group,numbers in review_groups.items():
    compact_review.append({'line_no':min(numbers),'group':group,'affected_lines':sorted(numbers),
        'priority':min(e['priority'] for n in numbers for e in events_by_line[n]),
        'checks':sorted({e['check'] for n in numbers for e in events_by_line[n]}),
        'decision':'','correct_mcIds':'','comment':'','reviewer':''})
compact_review.sort(key=lambda r:(r['priority'],r['line_no']))
remember('review_queue',compact_review)
print(f'Полная очередь: {len(review_events):,} событий, {len(events_by_line):,} строк, {len(compact_review):,} групп.')
control_groups,seen_controls = defaultdict(list),set()
for r in train:
    key = (r['duplicate_group'],r['labels'])
    if key not in seen_controls:
        control_groups[r['labels']].append(r); seen_controls.add(key)
control_sample = []
rng = random.Random(RANDOM_SEED)
for labels,group in sorted(control_groups.items()):
    for r in rng.sample(group,min(3,len(group))):
        control_sample.append({'line_no':r['line_no'],'item_id':r['item_id'],'labels':labels,
            'has_review_flags':r['line_no'] in events_by_line,
            'decision':'','correct_mcIds':'','comment':'','reviewer':''})
remember('control_sample',control_sample)
print('decision: оставить / исправить / исключить / уточнить. correct_mcIds: полный набор, например [102,106] или []. '
      'comment: обоснование фрагментом текста.')


Таблица: review_queue


line_no,group,affected_lines,priority,checks,decision,correct_mcIds,comment,reviewer
1,f28d81a1877ab3be35da3d3256d1e84a79f48374918cbcdcdc26c3f56381cd4c,[1],2,"['boundary_context', 'stock_noise']",,,,
78,2487acf6236d2699ef44289654ddfae1a7ea39e5d25be8e8c28b53d57cda66e8,[78],2,"['boundary_context', 'stock_noise']",,,,
93,24ca1a3f5d5989f29c84c6f21db0c4c08f2111c97da74dd0463b026cefc9e755,[93],2,"['keyword_without_label', 'third_party_context']",,,,
118,d707f35f22d35712fbcf056c27fafe6d892a0f330ebbff940edacc56da46b6fc,[118],2,"['seo_marker', 'third_party_context']",,,,
122,d96232c54a5f7fb98a08f948f580f85cb46d2f8b97e1a3a1cf12c18361ed6833,[122],2,"['boundary_context', 'stock_noise']",,,,
142,6026b8c31ea2e7e0291f72345f589d3c26887fdfdc6121a118ea88603dcea8ae,[142],2,"['mixed_alphabet', 'third_party_context']",,,,
167,773f4eba556078e6b625f2f9147a49da46290ddc27401317881ceb851730924f,"[167, 749, 1362, 1450, 1485, 1549, 1819, 1935, 1962, 2318, 2496, 3040, 3969, 4170, 4257, 4266, 4378, 4426, 4450, 4491, 4790, 4854, 4884, 5063, 5125, 7053, 7364, 8197, 8791, 9135, 9226, 9279, 9427, 9570, 9593, 9619, 9747, 9765, 9829, 9889]",2,"['boundary_context', 'keyword_without_label', 'negation_context']",,,,
190,e2b86e08ae0a3da138c0e6747178fc1ca1eef56fece60730a23e62e1dfe75d89,[190],2,"['boundary_context', 'negation_context', 'third_party_context']",,,,
275,8d7644ed16f59c56fd41001a82cfe86307826d44c6b0321787381c77aac1f5ec,[275],2,"['boundary_context', 'negation_context', 'third_party_context']",,,,
342,91d21d08cfd5dab353b1164e4e5bccf165bb2ae9925d106fc3719441c0310fc0,[342],2,"['boundary_context', 'mixed_alphabet', 'third_party_context']",,,,


Показано 15 из 3469 строк таблицы.
Полная очередь: 16,280 событий, 7,338 строк, 3,469 групп.
Таблица: control_sample


line_no,item_id,labels,has_review_flags,decision,correct_mcIds,comment,reviewer
9361,2508710500049,(),True,,,,
1204,2501241001650,(),False,,,,
224,2506137250754,(),True,,,,
5351,2503318250423,"(102,)",True,,,,
4888,2504178250254,"(102,)",True,,,,
4860,2514846751337,"(102,)",True,,,,
703,2521632001625,"(102, 103)",True,,,,
7107,2509690002109,"(102, 103)",True,,,,
115,2521711750522,"(102, 103)",True,,,,
9658,2502883750911,"(102, 103, 104)",True,,,,


Показано 15 из 92 строк таблицы.
decision: оставить / исправить / исключить / уточнить. correct_mcIds: полный набор, например [102,106] или []. comment: обоснование фрагментом текста.


## 15. Любые строки и полные тексты

Карточки раскрываются нажатием на заголовок. line_no — **физическая строка JSONL с единицы**, не item_id.
После Run All можно выполнять в новых ячейках:

~~~python
show_ad(1)                                      # полный текст конкретной строки
show_rows(start=20, limit=10)                    # следующая страница объявлений
show_check('label_without_keyword', start=0, limit=10)
show_table('duplicate_members', start=0, limit=50)
show_rows(start=0, limit=None)                   # ВСЕ объявления; большой вывод
~~~

Все проверки уже выполнены для полного файла независимо от числа карточек на экране.


In [15]:
show_rows(start=0,limit=3)
print('Доступные проверки:\n'+', '.join(checks))
print('\nДоступные таблицы:\n'+', '.join(tables))


Полные тексты: 3 из 10000, начало 0.


Таблица / полные объявления (HTML)

Таблица / полные объявления (HTML)

Таблица / полные объявления (HTML)

Доступные проверки:
parse, schema, extra_fields, duplicate_id, unusual_id, short_text, no_letters, exact_duplicates, exact_conflicts, normalized_conflicts, mixed_alphabet, replacement_character, html_tags, escaped_newline, stock_noise, seo_marker, prompt_like, emoji, link, phone, invisible, keyword_without_label, label_without_keyword, title_only, negation_context, third_party_context, tail_only, boundary_context, truncation_risk

Доступные таблицы:
file_info, field_types, lengths, longest, category_frequency, label_combinations, cooccurrence, duplicate_summary, duplicate_members, frequency_without_duplicates, keyword_rules, keyword_agreement, context_summary, boundary_summary, truncation, split_leakage, split_label_balance, review_queue, control_sample


## 16. Полные выгрузки и контроль охвата

checks.csv — правила/счётчики, findings.csv — **все срабатывания**, review_events.csv и review_queue.csv —
полная и сгруппированная очереди, control_sample.csv — контрольные примеры.
row_inventory.csv содержит **каждую физическую строку**, включая неразобранные, и статус проверки.
records_with_audit.jsonl — все успешно разобранные исходные объекты целиком рядом с результатами.
tables/ — все таблицы без ограничения экранного вывода. manifest.json фиксирует параметры и SHA-256.

Каждый запуск — новая папка с временем UTC, старые ручные решения не перезаписываются.
CSV защищены от интерпретации текстовых значений как формул; JSONL сохраняет исходные значения.


In [16]:
assert len({r['line_no'] for r in train})==N
assert sum(combinations.values())==sum(cardinality.values())==N
assert set(CATEGORIES)==set(KEYWORDS)
assert all(sum(row[k] for k in ['слово+метка','слово без метки','метка без слова','нет обоих'])==N for row in agreement)
assert all(1<=e['line_no']<=physical_lines and e['check'] in checks for e in findings)
for spec in checks.values():
    events = [e for e in findings if e['check']==spec['check']]
    assert spec['events']==len(events)
    assert spec['affected_rows']==len({e['line_no'] for e in events})
with TRAIN_PATH.open('rb') as stream:
    current_digest = hashlib.sha256()
    for block in iter(lambda:stream.read(1024*1024),b''): current_digest.update(block)
assert current_digest.hexdigest()==SOURCE_SHA256,'Файл изменился во время анализа. Перезапусти ноутбук.'
all_events_by_line = defaultdict(list)
for e in findings: all_events_by_line[e['line_no']].append(e)
parse_bad = {e['line_no'] for e in parse_issues}
row_inventory = [{'line_no':n,'item_id':by_line.get(n,{}).get('item_id'),
    'status':'parse_error' if n in parse_bad else 'schema_error' if n in bad_schema else 'analysed',
    'checks':sorted({e['check'] for e in all_events_by_line[n]}),'events':len(all_events_by_line[n])}
    for n in range(1,physical_lines+1)]
assert len(row_inventory)==physical_lines
manifest = {'run_id':RUN_ID,'source':str(TRAIN_PATH),'sha256':SOURCE_SHA256,'python':sys.version,
    'physical_lines':physical_lines,'parsed_objects':len(by_line),'schema_valid_rows':N,
    'parse_errors':len(parse_issues),'schema_error_rows':len(bad_schema),'random_seed':RANDOM_SEED,
    'semantic_review':'heuristic candidates, not manual ground truth',
    'scope':'train only; exact and normalized duplicates; no semantic near-duplicate scan',
    'checks':list(checks.values())}
def csv_cell(value):
    if isinstance(value,(list,tuple,dict,set)):
        value = json.dumps(sorted(value) if isinstance(value,set) else value,ensure_ascii=False)
    if value is None: return ''
    if isinstance(value,str) and value.lstrip().startswith(('=','+','-','@')):
        return "'"+value
    return value
def save_csv(path,rows,columns=None):
    rows = list(rows)
    columns = columns or list(dict.fromkeys(k for row in rows for k in row)) or ['line_no','reason']
    with path.open('w',encoding='utf-8-sig',newline='') as f:
        writer = csv.DictWriter(f,fieldnames=columns)
        writer.writeheader()
        writer.writerows({k:csv_cell(row.get(k,'')) for k in columns} for row in rows)
if EXPORT_RESULTS:
    OUTPUT_DIR.mkdir(parents=True,exist_ok=False)
    (OUTPUT_DIR/'tables').mkdir()
    for name,rows in [('checks',list(checks.values())),('findings',findings),('review_events',review_events),
        ('review_queue',compact_review),('control_sample',control_sample),('row_inventory',row_inventory),
        ('parse_issues',parse_issues),('schema_issues',schema_issues)]:
        save_csv(OUTPUT_DIR/f'{name}.csv',rows)
    for name,rows in tables.items(): save_csv(OUTPUT_DIR/'tables'/f'{name}.csv',rows)
    with (OUTPUT_DIR/'records_with_audit.jsonl').open('w',encoding='utf-8') as f:
        for n,row in sorted(by_line.items()):
            f.write(json.dumps({'line_no':n,'original':row,'schema_valid':n not in bad_schema,
                               'findings':all_events_by_line[n]},ensure_ascii=False)+'\n')
    (OUTPUT_DIR/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    print('Все результаты сохранены:',OUTPUT_DIR)
    table([{'file':str(p.relative_to(OUTPUT_DIR)),'bytes':p.stat().st_size}
           for p in sorted(OUTPUT_DIR.rglob('*')) if p.is_file()],limit=None)
else: print('Экспорт отключён; результаты остаются в переменных ноутбука.')
print('Проверки согласованности пройдены. Исходный файл не изменился.')


Все результаты сохранены: /home/Prakser/Projects/Innoglobalhack/dataset_review/train_audit_runs/20261114T084053_451436Z


file,bytes
checks.csv,8138
control_sample.csv,4164
findings.csv,23091821
manifest.json,13343
parse_issues.csv,19
records_with_audit.jsonl,74101143
review_events.csv,6796113
review_queue.csv,439520
row_inventory.csv,910323
schema_issues.csv,19


Показано 29 из 29 строк таблицы.
Проверки согласованности пройдены. Исходный файл не изменился.


## 17. Итог: что нормально, что исправлять и что проверять

Сводка рассчитывается при запуске. Ноль срабатываний означает лишь «данное правило не нашло проблему».
Корректная схема не гарантирует качество меток. Сначала проверь противоречия/контекст,
согласуй спорные случаи с командой, затем собери отдельную ручную валидацию.


In [17]:
remember('final_checks',[{'check':c['check'],'проверка':c['title'],'уровень':c['level'],
    'проверено единиц':c['checked'],'строк со срабатыванием':c['affected_rows'],'событий':c['events']}
    for c in checks.values()],limit=None)
error_rows = {e['line_no'] for e in findings if e['level']=='ОШИБКА'}
print(f'\nОХВАТ: {physical_lines:,} физических строк; {N:,} записей прошли структурную проверку.')
print(f'ОШИБКИ: {len(error_rows):,} строк с ошибкой чтения/схемы или противоречием точного входа.')
print(f'ПОВТОРЫ: {N-len(unique_exact):,} повторов сверх первого по точному заголовку и описанию.')
print(f'РУЧНАЯ ПРОВЕРКА: {len(events_by_line):,} строк; {len(compact_review):,} групп; контроль — {len(control_sample):,} примеров.')
print('\nНормально: [] разрешён; все 5 категорий допустимы при реальном предложении услуг; эмодзи не ошибка.')
print('Исправлять первым: ошибки формата и доказанные противоречия точных дубликатов, если найдены.')
print('Проверить вручную: отрицания, чужие услуги, складские хвосты, расхождения словаря и неоднозначные услуги.')
print('Перед обучением: согласовать метки, разделить группы дубликатов, измерить длины выбранным токенизатором.')
print('Не доказано: смысловая правильность всех меток, отсутствие почти одинаковых перефраз и качество на test.')
if EXPORT_RESULTS: save_csv(OUTPUT_DIR/'tables/final_checks.csv',tables['final_checks'])


Таблица: final_checks


check,проверка,уровень,проверено единиц,строк со срабатыванием,событий
parse,Читаемость всех строк JSONL,ОШИБКА,10000,0,0
schema,Соответствие структуре и справочнику,ОШИБКА,10000,0,0
extra_fields,Дополнительные поля,ИНФОРМАЦИЯ,10000,0,0
duplicate_id,Повторяющиеся item_id,ПРОВЕРИТЬ,10000,0,0
unusual_id,Подозрительные значения item_id,ПРОВЕРИТЬ,10000,0,0
short_text,Очень короткие описания,ПРОВЕРИТЬ,10000,236,236
no_letters,Текст без букв,ПРОВЕРИТЬ,10000,0,0
exact_duplicates,Полностью одинаковые входные тексты,РИСК УТЕЧКИ,10000,5400,5400
exact_conflicts,Противоречивые метки точных дубликатов,ОШИБКА,10000,0,0
normalized_conflicts,Противоречия после нормализации,ПРОВЕРИТЬ,10000,0,0


Показано 29 из 29 строк таблицы.

ОХВАТ: 10,000 физических строк; 10,000 записей прошли структурную проверку.
ОШИБКИ: 0 строк с ошибкой чтения/схемы или противоречием точного входа.
ПОВТОРЫ: 4,998 повторов сверх первого по точному заголовку и описанию.
РУЧНАЯ ПРОВЕРКА: 7,338 строк; 3,469 групп; контроль — 92 примеров.

Нормально: [] разрешён; все 5 категорий допустимы при реальном предложении услуг; эмодзи не ошибка.
Исправлять первым: ошибки формата и доказанные противоречия точных дубликатов, если найдены.
Проверить вручную: отрицания, чужие услуги, складские хвосты, расхождения словаря и неоднозначные услуги.
Перед обучением: согласовать метки, разделить группы дубликатов, измерить длины выбранным токенизатором.
Не доказано: смысловая правильность всех меток, отсутствие почти одинаковых перефраз и качество на test.
